In [ ]:
!pip install --upgrade --force-reinstall torch torch_geometric

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
from torch_geometric.data import Data, Dataset, DataLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import to_undirected
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
graph = torch.load(f'/content/drive/MyDrive/ma_files/data/synthetic_64_01/graphs_sequential.pt', weights_only=False)

In [ ]:
graph[:-2]

In [ ]:
# ------------ Model ------------

class GATShift(nn.Module):
    """
    GATv2-based node regression: predicts (dx, dy) per node.
    Uses edge_attr through GATv2Conv(edge_dim=...).
    """
    def __init__(self, in_dim, edge_dim, hidden=64, heads=4, layers=3, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList()
        self.activ = nn.ELU()
        self.dropout = nn.Dropout(dropout)

        # First
        self.layers.append(GATv2Conv(in_channels=in_dim, out_channels=hidden,
                                     heads=heads, edge_dim=edge_dim, dropout=dropout, concat=True))
        out_dim = hidden * heads

        # Middle
        for _ in range(layers - 2):
            self.layers.append(GATv2Conv(in_channels=out_dim, out_channels=hidden,
                                         heads=heads, edge_dim=edge_dim, dropout=dropout, concat=True))
            out_dim = hidden * heads

        # Last (set concat=False to keep output dim = hidden)
        self.layers.append(GATv2Conv(in_channels=out_dim, out_channels=hidden,
                                     heads=1, edge_dim=edge_dim, dropout=dropout, concat=False))

        # Regression head -> (dx, dy)
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 2)
        )

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        for i, gat in enumerate(self.layers):
            x = gat(x, edge_index, edge_attr=edge_attr)
            if i < len(self.layers) - 1:
                x = self.activ(x)
                x = self.dropout(x)
        return self.head(x)  # [N,2]

In [ ]:
# ------------ Training loop ------------

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = model(batch)

        # Mask: only synthetic_1 nodes (line_id == 1)
        line_ids = batch.x[:, 0]  # first column of features
        mask = (line_ids == 1)

        loss = F.mse_loss(pred[mask], batch.y[mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = model(batch)
            line_ids = batch.x[:, 0]
            mask = (line_ids == 1)
            loss = F.mse_loss(pred[mask], batch.y[mask])
            total_loss += loss.item()
    return total_loss / len(loader)


In [ ]:
def plot_graph_nodes(graph, pred_shift=None, title="Graph nodes"):
    coords = graph.x[:, 1:3].cpu().numpy()       # [N, 2] original coordinates
    line_ids = graph.x[:, 0].cpu().numpy()       # line identifiers (0=original, 1=synthetic_1, 2=synthetic_2)

    syn1_coords = coords[line_ids == 1]          # synthetic_1 (input)
    syn2_coords = coords[line_ids == 2]          # synthetic_2 (target)

    plt.figure(figsize=(6, 6))
    plt.scatter(syn1_coords[:, 0], syn1_coords[:, 1], label="Synthetic_1 (input)", alpha=0.7, s=10)
    plt.scatter(syn2_coords[:, 0], syn2_coords[:, 1], label="Synthetic_2 (target)", alpha=0.7, s=10)

    if pred_shift is not None:
        # Apply predicted shift only to synthetic_1 nodes
        pred_shift = pred_shift.cpu().numpy() if torch.is_tensor(pred_shift) else pred_shift
        shifted_coords = syn1_coords + pred_shift[line_ids == 1]
        plt.scatter(shifted_coords[:, 0], shifted_coords[:, 1],
                    label="Predicted (shifted)", marker="x", alpha=0.7)

    plt.title(title)
    plt.legend()
    plt.gca().set_aspect('equal', 'box')
    plt.show()

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def plot_graph_with_edges(graph, title="Graph with edges"):
    coords = graph.x[:, 1:3].cpu().numpy()
    edge_index = graph.edge_index.cpu().numpy()

    G = nx.Graph()
    for i in range(coords.shape[0]):
        G.add_node(i, pos=(coords[i, 0], coords[i, 1]))
    for src, dst in edge_index.T:
        G.add_edge(int(src), int(dst))

    pos = nx.get_node_attributes(G, 'pos')
    plt.figure(figsize=(6, 6))
    nx.draw(G, pos, node_size=30, node_color="blue", edge_color="gray", alpha=0.5)
    plt.title(title)
    plt.gca().set_aspect('equal', 'box')
    plt.show()


In [ ]:
if __name__ == "__main__":
    # Load your saved graphs
    graphs = torch.load('/content/drive/MyDrive/ma_files/data/synthetic_64_01/graphs_sequential_2.pt', weights_only=False)

    # Ensure we have a list (even if only one graph was saved)
    if not isinstance(graphs, list):
        graphs = [graphs]

    # Number of Subgraphs
    num_subgraphs = len(graphs)
    print(f"Number of Subgraphs: {num_subgraphs}")

    plot_graph_nodes(graph, title="Loaded Graph (Node Positions)")

    # DataLoader (batch_size=1 for variable-size graphs)
    train_loader = DataLoader(graphs[:100], batch_size=1, shuffle=True)

    # Use first graph for inferring dimensions
    sample = graphs[0]

    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Model
    model = GATShift(
        in_dim=sample.x.size(1),
        edge_dim=sample.edge_attr.size(1) if sample.edge_attr is not None else 0,
        hidden=64,
        heads=4,
        layers=3,
        dropout=0.1
    ).to(device)

    # Optimizer
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

    # Train loop
    for epoch in range(1, 11):
        tr = train_epoch(model, train_loader, opt, device)
        va = eval_epoch(model, train_loader, device)  # Using same data for validation
        if epoch % 10 == 0:
            print(f"epoch {epoch:02d} | train MSE {tr:.5f} | val MSE {va:.5f}")

    # Inference on first graph
    model.eval()
    with torch.no_grad():
        graph = graphs[0].to(device)
        pred_shift = model(graph).cpu().numpy()  # [N, 2]

        # Mask: only synthetic_1 nodes
        line_ids = graph.x[:, 0].cpu().numpy()
        mask = line_ids == 1

        corrected_coords = graph.x[mask, 1:3].cpu().numpy() + pred_shift[mask]

        print("Predicted first 3 shifts (synthetic_1 nodes only):", pred_shift[mask][:3])
        print("Corrected first 3 coordinates:", corrected_coords[:3])


In [ ]:
# Inference on one polyline:
model.eval()
with torch.no_grad():
    graph = graph.to(device)
    pred_shift = model(graph).cpu().numpy()  # [N, 2]

    # Extract node coordinates for synthetic_1 nodes
    line_ids = graph.x[:, 0].cpu().numpy()
    mask = line_ids == 1  # Only synthetic_1 nodes

    coords_syn1 = graph.x[mask, 1:3].cpu().numpy()  # shape [num_nodes_syn1, 2]

    shifted_coords = coords_syn1 + pred_shift[mask]  # Apply predicted shift

    print("Predicted first 3 shifts:", pred_shift[mask][:3])
    print("Corrected first 3 coordinates:", shifted_coords[:3])


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(coords_syn1[:, 0], coords_syn1[:, 1], label="Original synthetic_1", alpha=0.6)
plt.scatter(shifted_coords[:, 0], shifted_coords[:, 1], label="Corrected (predicted)", alpha=0.6)
plt.legend()
plt.gca().set_aspect('equal', 'box')
plt.show()